In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os

dataset_path = "/kaggle/input/mevid-sample"

for dirname, _, filenames in os.walk(dataset_path):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
ls

In [ ]:
!git clone https://github.com/Quang0309/MultiCameraTracking.git
!pip install -r MultiCameraTracking/requirements-train.txt
!git clone https://github.com/JDAI-CV/fast-reid.git
!cd fast-reid && pip install -r docs/requirements.txt

In [ ]:
cd fast-reid

In [ ]:
!mv /kaggle/working/fast-reid/fastreid /kaggle/working/MultiCameraTracking/

In [ ]:
ls


In [ ]:
!git pull

In [ ]:
!git fetch origin
!git reset --hard origin/main

In [ ]:
!wget https://github.com/JDAI-CV/fast-reid/releases/download/v0.1.1/msmt_agw_R50-ibn.pth -O /kaggle/working/msmt_agw_R50-ibn.pth

In [ ]:
!mkdir -p /kaggle/working/result
!cp -r /kaggle/input/datasets/quangnguyen97/agw-train-result-final/agw_train_3 /kaggle/working/result/

In [ ]:
#%cd ..
#%cd ..
#/kaggle/working/msmt_agw_R50-ibn.pth
%cd MultiCameraTracking/
#/kaggle/input/datasets/quangnguyen97/mevid-sample
!PYTHONPATH=. FASTREID_DATASETS=/kaggle/input/datasets/quangnguyen97/mevid-sample python scripts/train.py \
    --config configs/mevid_agw_r50_ibn.yml \
    DATASETS.NAMES "('MEVID_Mini',)" \
    DATASETS.TESTS "('MEVID_Mini',)" \
    DATALOADER.NUM_WORKERS 2 \
    SOLVER.IMS_PER_BATCH 64 \
    SOLVER.OPT "Adam" \
    SOLVER.BASE_LR 0.00035 \
    SOLVER.HEADS_LR_FACTOR 10.0 \
    SOLVER.WARMUP_ITERS 100 \
    SOLVER.FREEZE_ITERS 0 \
    SOLVER.AMP.ENABLED False \
    MODEL.LOSSES.CE.EPSILON 0.0 \
    SOLVER.CHECKPOINT_PERIOD 2 \
    MODEL.WEIGHTS "/kaggle/working/msmt_agw_R50-ibn.pth" \
    OUTPUT_DIR "/kaggle/working/result/agw_aggressive_run"

In [ ]:
!rm -f /kaggle/working/msmt_agw_R50-ibn.pth
!wget -L -O /kaggle/working/msmt_agw_R50-ibn.pth https://github.com/JDAI-CV/fast-reid/releases/download/v0.1.1/msmt_agw_R50-ibn.pth

In [ ]:
%cd MultiCameraTracking/
#/kaggle/input/datasets/quangnguyen97/mevid-sample
!PYTHONPATH=. FASTREID_DATASETS=/kaggle/input/datasets/quangnguyen97/mevid-sample python scripts/train.py \
    --config configs/mevid_agw_r50_ibn.yml \
    --eval-only \
    DATASETS.NAMES "('MEVID_Mini',)" \
    DATASETS.TESTS "('MEVID_Mini',)" \
    SOLVER.IMS_PER_BATCH 32 \
    DATALOADER.NUM_WORKERS 0 \
    MODEL.WEIGHTS "/kaggle/working/msmt_agw_R50-ibn.pth" \
    OUTPUT_DIR "/kaggle/working/result/agw"

In [ ]:
import shutil

# Compress /kaggle/working/result into /kaggle/working/result.zip
shutil.make_archive('/kaggle/working/result/agw_aggressive_run', 'zip', '/kaggle/working/result')

# Evaluate

In [ ]:
%cd MultiCameraTracking/
!PYTHONPATH=. FASTREID_DATASETS=/kaggle/input/datasets/quangnguyen97/mevid-sample python scripts/eval_mevid_protocols.py \
    --config configs/mevid_agw_r50_ibn.yml \
    --weights /kaggle/working/result/agw_train_3/model_best.pth \
    DATASETS.NAMES "('MEVID_Mini',)" \
    DATASETS.TESTS "('MEVID_Mini',)"

In [ ]:
%cd MultiCameraTracking/
!PYTHONPATH=. FASTREID_DATASETS=/kaggle/input/datasets/quangnguyen97/mevid-sample python scripts/eval_mevid_protocols.py \
    --config configs/mevid_agw_r50_ibn.yml \
    --weights /kaggle/working/msmt_agw_R50-ibn.pth \
    DATASETS.NAMES "('MEVID_Mini',)" \
    DATASETS.TESTS "('MEVID_Mini',)"